# 05 · Pilot: paraphrase (design doc §8.8.10)

**How to run**: pick a Colab server with a GPU (a T4 is enough) and click "Run All"; about 2.5 hours. If the connection drops, click "Run All" again; finished parts are skipped. Results go to `DATA_DIR/05_pilot_paraphrase/`.

**What it computes**: Qwen2.5-3B-Instruct paraphrases 100 texts from each of three sources: human originals, AI continuations at T = 0.8 (detectable) and AI continuations at T = 1.0 (blind spot). Each source is paraphrased twice, with greedy decoding and with plain sampling. GPT-2 XL and Qwen2.5-3B then score the paraphrases to see whether they can be told apart from the human originals.

In [ ]:
# 1. Check for a GPU.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to a GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("05_pilot_paraphrase", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # the models are public; skip the Colab secret lookup for HF_TOKEN (it times out when run from VS Code)

In [ ]:
# 3. Install packages and prepare 100 texts from each of the three sources.
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, traceback, time
import numpy as np, pandas as pd
humans = {h["id"]: h for h in json.load(open("01_pilot_decoding/humans.json"))}
t08 = {r["id"]: r for r in json.load(open("03_pilot_temperature/gen_T0.8.json"))}
t10 = {r["id"]: r for r in json.load(open("03_pilot_temperature/gen_T1.0.json"))}
SOURCES = {
    "human": {i: h["text"] for i, h in humans.items()},
    "ai_T0.8": {i: r["prefix"] + r["ai"] for i, r in t08.items()},
    "ai_T1.0": {i: r["prefix"] + r["ai"] for i, r in t10.items()},
}
print({k: len(v) for k, v in SOURCES.items()})

In [ ]:
# 4. Paraphrase: Qwen2.5-3B-Instruct, three sources x two decoding settings. Progress is saved every 10 texts; a rerun resumes where it stopped.
import token_info
PARA = "Qwen/Qwen2.5-3B-Instruct"
INSTRUCTION = ("Rewrite the following text in the same language. Keep the meaning, facts, named entities, numbers, "
               "negations and tone; do not add or remove information. Rephrase nouns, verbs, adjectives and adverbs "
               "naturally and you may restructure sentences, but do not change the meaning just to replace every word. "
               "Output only the rewritten text, with no explanation.")
DECODINGS = {"greedy": dict(paraphrase_do_sample=False), "sample": dict(paraphrase_do_sample=True, temperature=1.0, top_p=1.0)}

token_info.SETTINGS.update(scoring_model=PARA, sampling_model=PARA, prefix_tokens=30, paraphrase_enabled=True,
                           paraphrase_instruction=INSTRUCTION, paraphrase_max_new_tokens=512, save_token_features=False,
                           cache_dir="/content/hf", seed=42)
engine = None
for dec, opts in DECODINGS.items():
    for src, texts in SOURCES.items():
        path = f"05_pilot_paraphrase/para_{src}_{dec}.json"
        if os.path.exists(path):
            print("Already done, skipping:", path); continue
        part = path + ".part"   # progress file for a half-finished condition
        rows = json.load(open(part)) if os.path.exists(part) else []
        done = {r["id"] for r in rows}
        token_info.SETTINGS.update(**opts)
        try:
            engine = token_info.TokenInfo()   # settings are copied at construction, so rebuild it for each decoding
            start = time.time()
            for n, (i, text) in enumerate(texts.items()):
                if i in done:
                    continue
                engine.rng.manual_seed(42 + n)   # fixed seed per text: resuming gives the same result as one uninterrupted run
                try:
                    r = engine.paraphrase(text)
                except (ValueError, RuntimeError) as e:
                    print("Skipping text", i, ":", e); continue
                rows.append(dict(id=i, source=src, decoding=dec, original=text, paraphrase=r["text"],
                                 truncated=r.get("possibly_truncated", False)))
                if len(rows) % 10 == 0:
                    json.dump(rows, open(part, "w"), ensure_ascii=False)
                    print(f"  {src} {dec}: {len(rows)}/{len(texts)} texts, {(time.time() - start) / 60:.0f} min so far", flush=True)
            json.dump(rows, open(path, "w"), ensure_ascii=False)
            if os.path.exists(part): os.remove(part)
            print(src, dec, "done:", len(rows), "texts, truncated:", sum(r["truncated"] for r in rows))
        except Exception:
            traceback.print_exc(); print("This condition failed, skipping:", src, dec)
        finally:
            engine = None; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(f"/content/hf/models--{PARA.replace('/', '--')}", ignore_errors=True)

In [ ]:
# 5. Scoring of whole texts (human originals, the three source sets, six paraphrase sets) with GPT-2 XL and Qwen2.5-3B.
from transformers import AutoTokenizer, AutoModelForCausalLM

records = [dict(id=i, group="human", text=t) for i, t in SOURCES["human"].items()]
records += [dict(id=i, group="ai_T0.8", text=t) for i, t in SOURCES["ai_T0.8"].items()]
records += [dict(id=i, group="ai_T1.0", text=t) for i, t in SOURCES["ai_T1.0"].items()]
for f in sorted(glob.glob("05_pilot_paraphrase/para_*.json")):
    for r in json.load(open(f)):
        records.append(dict(id=r["id"], group=f"para_{r['source']}_{r['decoding']}", text=r["paraphrase"]))
print(pd.Series([r["group"] for r in records]).value_counts())

def score_texts(scorer_name, out_path):
    if os.path.exists(out_path):
        print("Already done, skipping:", out_path); return
    tok = AutoTokenizer.from_pretrained(scorer_name)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    model = AutoModelForCausalLM.from_pretrained(scorer_name, dtype=dtype).cuda().eval()
    out = []
    with torch.no_grad():
        for i, r in enumerate(records):
            ids = tok(r["text"], truncation=True, max_length=512)["input_ids"]
            if len(ids) < 16:
                continue
            x = torch.tensor([ids]).cuda()
            lp = torch.log_softmax(model(x).logits[0, :-1].float(), -1)
            p = lp.exp(); e1 = (p * lp).sum(-1)
            rows = [-lp.gather(1, x[0, 1:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2, p.topk(10, -1).values.sum(-1)]
            out.append(dict(id=r["id"], group=r["group"], scorer=scorer_name, arr=torch.stack(rows).cpu().numpy().astype(np.float32)))
            if i % 200 == 0:
                print(" ", scorer_name, i, "/", len(records), flush=True)
    pd.DataFrame(out).to_pickle(out_path)
    del model; gc.collect(); torch.cuda.empty_cache()

score_texts("gpt2-xl", "05_pilot_paraphrase/scores_gpt2-xl.pkl")
score_texts("Qwen/Qwen2.5-3B", "05_pilot_paraphrase/scores_Qwen2.5-3B.pkl")

In [ ]:
# 6. Quick summary: AUROC and margin of each group against the human originals. The full analysis is scripts/a12_paraphrase.py.
from sklearn.metrics import roc_auc_score
s = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob("05_pilot_paraphrase/scores_*.pkl"))], ignore_index=True)
s["surprisal"] = [a[0].mean() for a in s.arr]
s["fdg"] = [(a[1].sum() - a[0].sum()) / np.sqrt(a[2].sum()) for a in s.arr]
rows = []
for sc, d in s.groupby("scorer"):
    h = d[d.group == "human"]
    for g, x in d[d.group != "human"].groupby("group"):
        y = np.r_[np.ones(len(x)), np.zeros(len(h))]
        rows.append(dict(scorer=sc, group=g, n_texts=len(x), surprisal_margin=h.surprisal.mean() - x.surprisal.mean(),
                         AUROC=roc_auc_score(y, np.r_[x.fdg.values, h.fdg.values])))
table = pd.DataFrame(rows); table.to_csv("05_pilot_paraphrase/summary.csv", index=False)
print(table.round(3).to_string(index=False))

In [ ]:
# 7. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/05_pilot_paraphrase/")